In [29]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_RGB
import numpy as np
import time
import matplotlib.cm as cm

base = BaseOverlay("base.bit")
hdmi_out = base.video.hdmi_out

DISPLAY_W, DISPLAY_H = 1280, 720

mode = VideoMode(DISPLAY_W, DISPLAY_H, 24)
hdmi_out.configure(mode, PIXEL_RGB)
hdmi_out.start()

print("HDMI output started at 1280x720")

HDMI output started at 1280x720


In [30]:
# Use 1280x720 for full-resolution render.
# Use 640x360 for faster render, upscaled to 720p output.
# RENDER_W, RENDER_H = 640, 360
RENDER_W, RENDER_H = 1280, 720

def make_plane(width, height, center_r, center_i, zoom):
    view_w = 4.0 / zoom
    view_h = view_w * height / width

    x = np.linspace(center_r - view_w/2, center_r + view_w/2, width, dtype=np.float32)
    y = np.linspace(center_i - view_h/2, center_i + view_h/2, height, dtype=np.float32)

    return np.meshgrid(x, y)


def fractal_numpy(
    width,
    height,
    center_r=-0.5,
    center_i=0.0,
    zoom=1.0,
    max_iter=200,
    fractal="mandelbrot",
    julia_c=(-0.4, 0.6)
):
    Cr, Ci = make_plane(width, height, center_r, center_i, zoom)

    fractal = fractal.lower()

    if fractal == "julia":
        Zr = Cr.copy()
        Zi = Ci.copy()
        Cr = np.full_like(Zr, julia_c[0], dtype=np.float32)
        Ci = np.full_like(Zi, julia_c[1], dtype=np.float32)
    else:
        Zr = np.zeros_like(Cr, dtype=np.float32)
        Zi = np.zeros_like(Ci, dtype=np.float32)

    iter_count = np.full((height, width), max_iter, dtype=np.uint16)
    active = np.ones((height, width), dtype=bool)

    for n in range(max_iter):
        if not active.any():
            break

        zr = Zr[active]
        zi = Zi[active]
        cr = Cr[active]
        ci = Ci[active]

        if fractal in ("mandelbrot", "julia"):
            zr_new = zr*zr - zi*zi + cr
            zi_new = 2.0*zr*zi + ci

        elif fractal == "burning_ship":
            zr_abs = np.abs(zr)
            zi_abs = np.abs(zi)
            zr_new = zr_abs*zr_abs - zi_abs*zi_abs + cr
            zi_new = 2.0*zr_abs*zi_abs + ci

        elif fractal == "tricorn":
            zr_new = zr*zr - zi*zi + cr
            zi_new = -2.0*zr*zi + ci

        elif fractal == "multibrot3":
            # z = z^3 + c
            zr2 = zr*zr
            zi2 = zi*zi
            zr_new = zr*zr2 - 3.0*zr*zi2 + cr
            zi_new = 3.0*zr2*zi - zi*zi2 + ci

        else:
            raise ValueError(f"Unknown fractal type: {fractal}")

        mag_sq = zr_new*zr_new + zi_new*zi_new
        escaped_now = mag_sq > 4.0

        active_y, active_x = np.where(active)

        escaped_y = active_y[escaped_now]
        escaped_x = active_x[escaped_now]

        iter_count[escaped_y, escaped_x] = n + 1

        still_active = ~escaped_now

        update_y = active_y[still_active]
        update_x = active_x[still_active]

        Zr[update_y, update_x] = zr_new[still_active]
        Zi[update_y, update_x] = zi_new[still_active]

        active[escaped_y, escaped_x] = False

    return iter_count


def colourise_fast(iter_count, max_iter, cmap_name="twilight_shifted"):
    cmap = cm.get_cmap(cmap_name, max_iter + 1)
    lut = (cmap(np.arange(max_iter + 1))[:, :3] * 255).astype(np.uint8)

    rgb = lut[iter_count]
    rgb[iter_count == max_iter] = 0

    return rgb


def upscale_to_720p(rgb):
    h, w, _ = rgb.shape

    if w == DISPLAY_W and h == DISPLAY_H:
        return rgb

    x_idx = np.linspace(0, w - 1, DISPLAY_W).astype(np.int32)
    y_idx = np.linspace(0, h - 1, DISPLAY_H).astype(np.int32)

    return rgb[y_idx[:, None], x_idx[None, :]]


def write_hdmi(rgb):
    frame = hdmi_out.newframe()
    frame[...] = upscale_to_720p(rgb)
    hdmi_out.writeframe(frame)


class FractalViewer:
    def __init__(
        self,
        fractal="mandelbrot",
        center_r=-0.5,
        center_i=0.0,
        zoom=1.0,
        max_iter=200,
        julia_c=(-0.4, 0.6),
        cmap_name="twilight_shifted"
    ):
        self.fractal = fractal
        self.center_r = center_r
        self.center_i = center_i
        self.zoom = zoom
        self.max_iter = max_iter
        self.julia_c = julia_c
        self.cmap_name = cmap_name

    def render(self):
        t0 = time.perf_counter()

        escape = fractal_numpy(
            RENDER_W,
            RENDER_H,
            center_r=self.center_r,
            center_i=self.center_i,
            zoom=self.zoom,
            max_iter=self.max_iter,
            fractal=self.fractal,
            julia_c=self.julia_c
        )

        t_render = time.perf_counter() - t0

        t0 = time.perf_counter()
        rgb = colourise_fast(escape, self.max_iter, self.cmap_name)
        t_colour = time.perf_counter() - t0

        write_hdmi(rgb)

        mpix = (RENDER_W * RENDER_H) / 1e6

        print(
            f"{self.fractal:12s} "
            f"centre=({self.center_r:.8f}, {self.center_i:.8f}) "
            f"zoom={self.zoom:.2f} "
            f"iter={self.max_iter}"
        )
        print(f"Render: {t_render*1000:7.0f} ms   ({mpix/t_render:.2f} Mpix/s)")
        print(f"Colour: {t_colour*1000:7.0f} ms")

    def zoom_to(self, center_r, center_i, zoom=None, max_iter=None):
        self.center_r = center_r
        self.center_i = center_i

        if zoom is not None:
            self.zoom = zoom

        if max_iter is not None:
            self.max_iter = max_iter

        self.render()

    def zoom_in(self, factor=2.0):
        self.zoom *= factor
        self.render()

    def zoom_out(self, factor=2.0):
        self.zoom /= factor
        self.render()

    def pan(self, dx_frac=0.0, dy_frac=0.0):
        view_w = 4.0 / self.zoom
        view_h = view_w * RENDER_H / RENDER_W

        self.center_r += dx_frac * view_w
        self.center_i += dy_frac * view_h

        self.render()

    def set_iter(self, max_iter):
        self.max_iter = max_iter
        self.render()

In [27]:
# Mandelbrot - full view
viewer = FractalViewer(
    fractal="mandelbrot",
    center_r=-0.5,
    center_i=0.0,
    zoom=1.0,
    max_iter=200,
    cmap_name="twilight_shifted"
)

viewer.render()

tricorn      centre=(0.00000000, 0.00000000) zoom=1.000 iter=200
Render:   29877 ms   (0.03 Mpix/s)
Colour:     315 ms


In [ ]:
# Mandelbrot - seahorse valley
viewer = FractalViewer(
    fractal="mandelbrot",
    center_r=-0.743643887,
    center_i=0.131825904,
    zoom=250,
    max_iter=300,
    cmap_name="twilight_shifted"
)

viewer.render()

In [32]:
# Mandelbrot - elephant valley
viewer = FractalViewer(
    fractal="mandelbrot",
    center_r=0.275,
    center_i=0.0,
    zoom=45,
    max_iter=250,
    cmap_name="turbo"
)

viewer.render()

mandelbrot   centre=(0.27500000, 0.00000000) zoom=45.00 iter=250
Render:  162861 ms   (0.01 Mpix/s)
Colour:     477 ms


In [31]:
# Mandelbrot - spiral region
viewer = FractalViewer(
    fractal="mandelbrot",
    center_r=-0.761574,
    center_i=-0.0847596,
    zoom=1200,
    max_iter=500,
    cmap_name="magma"
)

viewer.render()

mandelbrot   centre=(-0.76157400, -0.08475960) zoom=1200.00 iter=500
Render:  156200 ms   (0.01 Mpix/s)
Colour:     311 ms


In [33]:
# Julia - classic
viewer = FractalViewer(
    fractal="julia",
    center_r=0.0,
    center_i=0.0,
    zoom=1.2,
    max_iter=250,
    julia_c=(-0.4, 0.6),
    cmap_name="twilight_shifted"
)

viewer.render()

julia        centre=(0.00000000, 0.00000000) zoom=1.20 iter=250
Render:   38054 ms   (0.02 Mpix/s)
Colour:     293 ms


In [34]:
# Julia - dendrite
viewer = FractalViewer(
    fractal="julia",
    center_r=0.0,
    center_i=0.0,
    zoom=1.3,
    max_iter=250,
    julia_c=(0.0, 1.0),
    cmap_name="turbo"
)

viewer.render()

julia        centre=(0.00000000, 0.00000000) zoom=1.30 iter=250
Render:    6018 ms   (0.15 Mpix/s)
Colour:     277 ms


In [35]:
# Julia - spiral
viewer = FractalViewer(
    fractal="julia",
    center_r=0.0,
    center_i=0.0,
    zoom=1.5,
    max_iter=300,
    julia_c=(-0.70176, -0.3842),
    cmap_name="plasma"
)

viewer.render()

julia        centre=(0.00000000, 0.00000000) zoom=1.50 iter=300
Render:   24692 ms   (0.04 Mpix/s)
Colour:     281 ms


In [36]:
# Julia - fragmented dust
viewer = FractalViewer(
    fractal="julia",
    center_r=0.0,
    center_i=0.0,
    zoom=1.1,
    max_iter=250,
    julia_c=(-0.8, 0.156),
    cmap_name="inferno"
)

viewer.render()

julia        centre=(0.00000000, 0.00000000) zoom=1.10 iter=250
Render:   43768 ms   (0.02 Mpix/s)
Colour:     311 ms


In [37]:
# Julia - fine structure
viewer = FractalViewer(
    fractal="julia",
    center_r=0.0,
    center_i=0.0,
    zoom=1.7,
    max_iter=300,
    julia_c=(0.285, 0.01),
    cmap_name="twilight"
)

viewer.render()

julia        centre=(0.00000000, 0.00000000) zoom=1.70 iter=300
Render:   41007 ms   (0.02 Mpix/s)
Colour:     299 ms


In [38]:
# Burning Ship - full view
viewer = FractalViewer(
    fractal="burning_ship",
    center_r=-0.5,
    center_i=-0.5,
    zoom=1.0,
    max_iter=200,
    cmap_name="hot"
)

viewer.render()

burning_ship centre=(-0.50000000, -0.50000000) zoom=1.00 iter=200
Render:   57783 ms   (0.02 Mpix/s)
Colour:     356 ms


In [39]:
# Burning Ship - main ship
viewer = FractalViewer(
    fractal="burning_ship",
    center_r=-1.75,
    center_i=-0.03,
    zoom=80,
    max_iter=300,
    cmap_name="inferno"
)

viewer.render()

burning_ship centre=(-1.75000000, -0.03000000) zoom=80.00 iter=300
Render:  104027 ms   (0.01 Mpix/s)
Colour:     337 ms


In [41]:
# Burning Ship - antenna region
viewer = FractalViewer(
    fractal="burning_ship",
    center_r=-1.744335,
    center_i=-0.017451,
    zoom=1200,
    max_iter=500,
    cmap_name="magma"
)

viewer.render()

burning_ship centre=(-1.74433500, -0.01745100) zoom=1200.00 iter=500
Render:  345655 ms   (0.00 Mpix/s)
Colour:     451 ms


In [42]:
# Burning Ship - deep detail
viewer = FractalViewer(
    fractal="burning_ship",
    center_r=-1.75895,
    center_i=-0.02991,
    zoom=900,
    max_iter=450,
    cmap_name="turbo"
)

viewer.render()

burning_ship centre=(-1.75895000, -0.02991000) zoom=900.00 iter=450
Render:   28352 ms   (0.03 Mpix/s)
Colour:     277 ms


In [43]:
# Tricorn - full view
viewer = FractalViewer(
    fractal="tricorn",
    center_r=0.0,
    center_i=0.0,
    zoom=1.0,
    max_iter=200,
    cmap_name="twilight_shifted"
)

viewer.render()

tricorn      centre=(0.00000000, 0.00000000) zoom=1.00 iter=200
Render:   29400 ms   (0.03 Mpix/s)
Colour:     317 ms


In [44]:
# Tricorn - central detail
viewer = FractalViewer(
    fractal="tricorn",
    center_r=0.0,
    center_i=0.0,
    zoom=3.0,
    max_iter=250,
    cmap_name="turbo"
)

viewer.render()

tricorn      centre=(0.00000000, 0.00000000) zoom=3.00 iter=250
Render:  109539 ms   (0.01 Mpix/s)
Colour:     406 ms


In [45]:
# Tricorn - side lobe - coords wrong or something
viewer = FractalViewer(
    fractal="tricorn",
    center_r=-0.25,
    center_i=0.65,
    zoom=12,
    max_iter=300,
    cmap_name="plasma"
)

viewer.render()

tricorn      centre=(-0.25000000, 0.65000000) zoom=12.00 iter=300
Render:    3569 ms   (0.26 Mpix/s)
Colour:     280 ms


In [46]:
# Tricorn - upper structure
viewer = FractalViewer(
    fractal="tricorn",
    center_r=0.35,
    center_i=0.55,
    zoom=20,
    max_iter=350,
    cmap_name="magma"
)

viewer.render()

tricorn      centre=(0.35000000, 0.55000000) zoom=20.00 iter=350
Render:  267715 ms   (0.00 Mpix/s)
Colour:     506 ms


In [47]:
# Multibrot 3 - full view
# Not actually going to be implemented into hardware, but may as well try it out
viewer = FractalViewer(
    fractal="multibrot3",
    center_r=0.0,
    center_i=0.0,
    zoom=1.0,
    max_iter=200,
    cmap_name="twilight_shifted"
)

viewer.render()

multibrot3   centre=(0.00000000, 0.00000000) zoom=1.00 iter=200
Render:   61627 ms   (0.01 Mpix/s)
Colour:     356 ms


In [ ]:
# Multibrot 3 - zoomed centre
viewer = FractalViewer(
    fractal="multibrot3",
    center_r=0.0,
    center_i=0.0,
    zoom=3.0,
    max_iter=250,
    cmap_name="turbo"
)

viewer.render()

# Basic Zoom/Pan commands

In [ ]:
# zoom in
viewer.zoom_in(2.0)

In [ ]:
# zoom out
viewer.zoom_out(2.0)

In [ ]:
# pan right
viewer.pan(dx_frac=0.25, dy_frac=0.0)

In [ ]:
# pan left
viewer.pan(dx_frac=-0.25, dy_frac=0.0)

In [ ]:
# pan up
viewer.pan(dx_frac=0.0, dy_frac=0.25)

In [ ]:
# pan down
viewer.pan(dx_frac=0.0, dy_frac=0.25)

In [ ]:
# increase detail
viewer.set_iter(viewer.max_iter + 100)

In [ ]:
# decrease detail
viewer.set_iter(max(50, viewer.max_iter - 100))

In [28]:
# cleanup
hdmi_out.stop()
del hdmi_out